# 16. Mini Classroom Chat

**Topic:** Multi-Client Socket Server

## Aim
Run a local chat server that broadcasts messages to connected clients. Use separate terminals for clients.

## Algorithm
1. Import the required Python modules.
2. Accept or prepare the required input/data.
3. Process the data using the concept **Multi-Client Socket Server**.
4. Display the result and handle common errors.

## Program
Run the code cell below. Some networking examples are limited to localhost or authorized test systems.


In [ ]:
import socket
import threading

HOST, PORT = "127.0.0.1", 50008
clients = set()
clients_lock = threading.Lock()

def broadcast(message, sender=None):
    with clients_lock:
        targets = list(clients)
    for client in targets:
        if client is sender:
            continue
        try:
            client.sendall(message)
        except OSError:
            with clients_lock:
                clients.discard(client)

def handle_client(conn, address):
    print("Connected:", address)
    with clients_lock:
        clients.add(conn)
    try:
        conn.sendall(b"Welcome! Type messages; use /quit to leave.\\n")
        while True:
            data = conn.recv(1024)
            if not data:
                break
            if data.strip() == b"/quit":
                break
            broadcast(f"{address[1]}: ".encode() + data, sender=conn)
    except OSError:
        pass
    finally:
        with clients_lock:
            clients.discard(conn)
        conn.close()
        print("Disconnected:", address)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((HOST, PORT))
    server.listen()
    print(f"Chat server on {HOST}:{PORT}. Press Ctrl+C to stop.")
    try:
        while True:
            conn, address = server.accept()
            threading.Thread(target=handle_client, args=(conn, address), daemon=True).start()
    except KeyboardInterrupt:
        print("\\nServer stopping.")